# Smart Developer Retrieval Demo

This notebook demonstrates the current retrieval prototype for the Smart Developer project.

The goal is to retrieve property sites that best match a development strategy, then explain why the retrieved sites appear relevant.

This demo uses:
- a property-level site feature bundle
- a heuristic multi-strategy scoring layer
- a tuned two-tower retrieval model
- fusion reranking
- local explanation generation

The notebook is intended as a qualitative product demo rather than a training notebook.

## System Overview

The current retrieval stack works as follows:

1. **Site representation**
   Each property polygon is enriched with zoning, lot-size, constraint, and transport-access features.

2. **Heuristic strategy scoring**
   Each site receives a first-pass score for strategies such as:
   - single dwelling rebuild
   - granny flat
   - dual occupancy
   - townhouse / multi-dwelling
   - low-rise apartment
   - land bank / hold
   - assembly opportunity

3. **Two-tower retrieval**
   A tuned bi-encoder / two-tower model embeds:
   - the strategy query
   - the candidate site text

4. **Fusion reranking**
   Retrieval similarity is combined with the heuristic strategy score to produce a final ranking.

5. **Explanation generation**
   A local explanation pipeline generates a short rationale for each retrieved site.

In [2]:
from pathlib import Path
import pandas as pd

from algorithm.src.retrieval.hybrid_retrieve import HybridRetriever, RetrievalRequest

## Load the Retriever

We use the tuned `two_tower_v1` model as the default retrieval backbone for this demo.

The retriever will:
- encode the query
- retrieve candidate sites
- rerank them using fusion
- deduplicate near-duplicate address records
- optionally attach explanations

In [3]:
retriever = HybridRetriever(experiment="two_tower_v1")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
def demo_columns(strategy: str) -> list[str]:
    score_col = f"{strategy}_score"
    cols = [
        "RID",
        "address",
        "primary_zoning_code",
        "lot_size_band",
        "constraint_severity_band",
        "station_distance_band",
        "top_strategy",
        score_col,
        "retrieval_similarity",
        "fusion_score",
        "explanation",
    ]
    return cols


def run_demo_query(
    strategy: str,
    query_text: str,
    top_k: int = 5,
    recall_k: int = 200,
    alpha: float = 0.5,
    beta: float = 0.5,
) -> pd.DataFrame:
    request = RetrievalRequest(
        strategy=strategy,
        query_text=query_text,
        top_k=top_k,
        recall_k=recall_k,
        alpha=alpha,
        beta=beta,
        dedupe_by_address=True,
        attach_explanations=True,
    )
    results = retriever.retrieve(request)
    cols = [c for c in demo_columns(strategy) if c in results.columns]
    return results[cols].copy()

## Demo Query 1 — Low-Rise Apartment

This query represents a higher-intensity urban redevelopment intent.

Expected retrieval pattern:
- higher-development zoning
- larger site size
- strong rail / metro access
- relatively low current constraint burden

In [5]:
apartment_query = (
    "strategy low_rise_apartment | "
    "higher intensity redevelopment | "
    "high_dev zoning preferred | "
    "lot size preference xl | "
    "strong station access preferred"
)

apartment_results = run_demo_query(
    strategy="low_rise_apartment",
    query_text=apartment_query,
    top_k=5,
)

print(apartment_results[["address", "explanation"]])

                                            address  \
21144  109/298-312 PENNANT HILLS ROAD PENNANT HILLS   
13254         33/91A-97 LONGFIELD STREET CABRAMATTA   
45050                   27/61 WEST PARADE WEST RYDE   
38486              30/181 PACIFIC HIGHWAY LINDFIELD   
38451                  510B/3 TIMBROL AVENUE RHODES   

                                             explanation  
21144  This development strategy, focused on low-rise...  
13254  This development strategy, focused on low-rise...  
45050  This development strategy, focused on low-rise...  
38486  This development strategy, focused on low-rise...  
38451  This development strategy of low-rise apartmen...  


### Observation

For this query, we expect the retriever to prioritise:
- urban or centre-style zoning
- large sites
- strong transport access
- low constraint severity

This is the clearest example of the system retrieving high-intensity redevelopment candidates.

## Demo Query 2 — Dual Occupancy

This query represents a lower-intensity infill strategy than apartments.

Expected retrieval pattern:
- lower- or medium-density residential context
- moderate-to-large lot size
- lower constraint burden
- station access helpful but not essential

In [6]:
dual_occ_query = (
    "strategy dual_occupancy | "
    "residential infill | "
    "prefers R1 R2 R3 R5 style context | "
    "moderate to large lot size | "
    "access not critical"
)

dual_occ_results = run_demo_query(
    strategy="dual_occupancy",
    query_text=dual_occ_query,
    top_k=5,
)

dual_occ_results

,RID,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,dual_occupancy_score,retrieval_similarity,fusion_score,explanation
38012,7439959,102/28 YOUNG STREET RANDWICK,R1,xl,low,800m_2km,dual_occupancy,74.8,0.727937,0.928161,This site has a good fit for the dual occupanc...
16481,7245479,103/1 GEORGE JULIUS AVENUE ZETLAND,R1,xl,low,800m_2km,dual_occupancy,74.8,0.727937,0.928161,This site has a good fit for the dual occupanc...
20781,6770158,108/255 MORRISON ROAD RYDE,R1,xl,low,800m_2km,dual_occupancy,74.8,0.727937,0.928161,This site has a good fit for the dual occupanc...
390,411249,61/7 BLACK LION PLACE KENSINGTON,R1,xl,low,800m_2km,dual_occupancy,74.8,0.727937,0.928161,This site has a good fit for the dual occupanc...
49477,1735682,59/95 BROMPTON ROAD KENSINGTON,R1,xl,low,800m_2km,dual_occupancy,74.8,0.727937,0.928161,This site has a good fit for the dual occupanc...


### Observation

For this query, we expect the retriever to return sites that look more like residential infill opportunities rather than apartment-scale urban redevelopment.

Compared with the apartment query, these results should generally appear:
- less intense
- more residential
- less dependent on strong station proximity

## Demo Query 3 — Land Bank / Hold

This query represents a more strategic and longer-horizon development intent.

Expected retrieval pattern:
- planning or location upside
- larger sites helpful
- transport access helpful
- some current friction may still be acceptable

In [7]:
land_bank_query = (
    "strategy land_bank_hold | "
    "future upside | "
    "zoning preference high_dev or mid_dev | "
    "lot size preference m l xl | "
    "access preference within_800m 2km 5km | "
    "constraints tolerated to some extent"
)

land_bank_results = run_demo_query(
    strategy="land_bank_hold",
    query_text=land_bank_query,
    top_k=5,
)

land_bank_results

,RID,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,top_strategy,land_bank_hold_score,retrieval_similarity,fusion_score,explanation
25717,5329143,42/23 BOOTH STREET WESTMEAD,E3,l,low,800m_2km,land_bank_hold,84.6,0.817654,0.842491,"This development strategy, land bank hold, sho..."
21570,6414038,309/8 MERRIVILLE ROAD KELLYVILLE RIDGE,E3,l,low,800m_2km,assembly_opportunity,84.6,0.817654,0.842491,"This development strategy, land bank hold, sho..."
4437,4480643,3/20 CRESCENT STREET HOLROYD,E3,l,low,800m_2km,land_bank_hold,84.6,0.817654,0.842491,"This development strategy, land bank hold, sho..."
15227,506588,27/401 PACIFIC HIGHWAY ARTARMON,E3,l,low,800m_2km,land_bank_hold,84.6,0.817654,0.842491,"This development strategy, land bank hold, sho..."
16858,757468,36-40 PARRAMATTA ROAD CROYDON,E3,l,low,800m_2km,land_bank_hold,84.6,0.817654,0.842491,"This development strategy, focusing on land ba..."


### Observation

For this query, the system may return sites that are not the most immediate build candidates, but still look strategically interesting because of their planning context, size, or accessibility.

This is a useful example of retrieval going beyond direct “ready to build now” logic.

## Side-by-Side Strategy Contrast

The three demo queries illustrate different retrieval behaviours:

- **Low-rise apartment**
  prioritises higher-intensity urban redevelopment conditions

- **Dual occupancy**
  prioritises lower-intensity residential infill conditions

- **Land bank / hold**
  prioritises longer-term strategic upside rather than immediate development fit

This contrast is important because the system is not trying to retrieve a single generic “good site”, but sites that are good for a specific development strategy.

In [8]:
summary_rows = []

for name, df, strategy in [
    ("low_rise_apartment", apartment_results, "low_rise_apartment"),
    ("dual_occupancy", dual_occ_results, "dual_occupancy"),
    ("land_bank_hold", land_bank_results, "land_bank_hold"),
]:
    score_col = f"{strategy}_score"
    summary_rows.append(
        {
            "demo_query": name,
            "mean_strategy_score": df[score_col].mean(),
            "mean_fusion_score": df["fusion_score"].mean(),
            "top_strategy_match_rate": (df["top_strategy"] == strategy).mean(),
        }
    )

pd.DataFrame(summary_rows)

,demo_query,mean_strategy_score,mean_fusion_score,top_strategy_match_rate
0,low_rise_apartment,97.5,1.000000,1.0
1,dual_occupancy,74.8,0.928161,1.0
2,land_bank_hold,84.6,0.842491,0.8


## Free-Text Query to Explanation Demo

This section demonstrates a simple end-to-end interaction:
- input a free-text development intent
- retrieve the most relevant site candidates
- generate a local explanation for the top result

This is useful for showing how the retrieval and explanation layers work together in practice.

In [9]:
free_text_query = (
    "I want a site for low-rise apartment redevelopment near a train station, "
    "with high development zoning, a large site, and limited planning constraints."
)

strategy = "low_rise_apartment"

free_text_results = run_demo_query(
    strategy=strategy,
    query_text=free_text_query,
    top_k=3,
)

free_text_results[[
    "address",
    "primary_zoning_code",
    "lot_size_band",
    "constraint_severity_band",
    "station_distance_band",
    "fusion_score",
    "explanation",
]]

,address,primary_zoning_code,lot_size_band,constraint_severity_band,station_distance_band,fusion_score,explanation
9125,16-20 PARK AVENUE WAITARA,R4,xl,low,within_800m,0.848648,"This development strategy, focused on low-rise..."
11435,405/4 ROOSEVELT AVENUE RIVERWOOD,R4,xl,low,within_800m,0.848648,"This development strategy, focused on low-rise..."
36846,12/1 CURTIS STREET CARINGBAH,R4,xl,low,within_800m,0.848648,"This development strategy, focused on low-rise..."
